In [1]:
from utils import *
#import episcanpy.api as epi
import time
import umap

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata

# Read data

In [2]:
atac = sc.read_h5ad("../../h5ad_files/c_atac_spatial_peaks.h5ad")
expr = sc.read_h5ad("../../h5ad_files/c_sp_trans_wide_ad.h5ad")

In [4]:
atac = atac[:, atac.var['n_cells_by_counts'] > 10].copy()

In [5]:
sc.pp.normalize_total(expr, target_sum=1e4)
sc.pp.log1p(expr)

#count_mat = atac.X.todense().T
count_mat = atac.X.T
tf_mat = 1.0 * count_mat / np.tile(np.sum(count_mat,axis=0), (count_mat.shape[0],1))
ATAC_count = np.log(1 + np.multiply(1e4*tf_mat,  np.tile((1.0 * count_mat.shape[1] / np.sum(count_mat,axis=1)).reshape(-1,1), (1,count_mat.shape[1]))))
atac.X = scipy.sparse.csc_matrix(ATAC_count.T)

atac, expr

(AnnData object with n_obs × n_vars = 5274 × 72148
     obs: 'sample_id', 'slice_id', 'class_label', 'subclass', 'label', 'cell_id', 'centroid_x', 'centroid_y', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_5_genes', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_50_genes', 'uniform_density', 'rna_count_based_density'
     var: 'chrom', 'start', 'end', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'is_training'
     uns: 'files', 'gearyC', 'spatial_neighbors'
     obsm: 'spatial'
     obsp: 'spatial_connectivities', 'spatial_distances',
 AnnData object with n_obs × n_vars = 5274 × 18931
     obs: 'sample_id', 'slice_id', 'class_label', 'subclass', 'label', 'cell_id', 'centroid_x', 'centroid_y', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_5_genes', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_50_genes', 'uniform_density',

In [6]:
atac.write_h5ad("desc_normalized_atac_peaks.h5ad")

In [7]:
expr.write_h5ad("desc_normalized_rna.h5ad")

In [6]:
#atac = sc.read_h5ad("desc_normalized_atac_peaks.h5ad")
#expr = sc.read_h5ad("desc_normalized_rna.h5ad")

In [7]:
#filtered_atac = atac[:, atac.var['n_cells_by_counts'] > 10].copy()

In [8]:
#atac = filtered_atac

In [8]:
np.all(atac.obs_names == expr.obs_names)

True

# Run Descart

In [9]:
save_path = 'result/first_descart_test'
if not os.path.exists(save_path):
    os.makedirs(save_path)
seed_base = 1
tf = None
pc = 10
k = 20
similarity = 'cosine'
iter_time = 4
spmethod = 'threshold'
neighbor = 5
sp_dist = 'recip'
pre_select = 'highest'
peaks_num = 50000
distance = 'euclidean'
r = 0.4

num_select_peak = 20000
idx_atac, _, _, _, _, _,_, _ = run_descart(atac, num_select_peak, seed_base=seed_base, tfidf=tf, ifPCA=True, pc=pc, k=k, similarity=similarity, iters=iter_time, spmethod=spmethod,neighbor=neighbor,sp_dist=sp_dist, pre_select=pre_select, peaks_num=peaks_num, distance=distance,r=r)

num_select_peak = 2000
idx_expr, _, _, _, _, _,_, _ = run_descart(expr, num_select_peak, seed_base=seed_base, tfidf=tf, ifPCA=True, pc=pc, k=k, similarity=similarity, iters=iter_time, spmethod=spmethod,neighbor=neighbor,sp_dist=sp_dist, pre_select=pre_select, peaks_num=peaks_num, distance=distance,r=r)

idx_atac.shape, idx_expr.shape

AnnData object with n_obs × n_vars = 5274 × 72148
    obs: 'sample_id', 'slice_id', 'class_label', 'subclass', 'label', 'cell_id', 'centroid_x', 'centroid_y', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_5_genes', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_50_genes', 'uniform_density', 'rna_count_based_density'
    var: 'chrom', 'start', 'end', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'is_training'
    uns: 'files', 'gearyC', 'spatial_neighbors'
    obsm: 'spatial'
    obsp: 'spatial_connectivities', 'spatial_distances'
(5274, 72148)
(5274, 5274)
(5274, 10)
0.0
compute scores


/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:258: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:277: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(


scores:
[10728.24689805 10728.31193408 10728.51018604 ... 30497.87156707
 33102.01575379 35579.58407506]
(5274, 5274)
(5274, 10)
0.0
compute scores


/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:258: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:277: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(


scores:
[11064.60772145 11064.68867038 11065.02940229 ... 30356.41584726
 33129.58528118 35233.49867013]
(5274, 5274)
(5274, 10)
0.0
compute scores


...

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:258: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:277: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(


scores:
[18855.09643351 18856.00791133 18857.67338159 ... 56134.28432648
 56555.1036081  56766.94185052]
(5274, 5274)
(5274, 10)
0.0
compute scores


/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:258: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:277: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(


scores:
[18855.06883036 18856.50122943 18857.74214495 ... 56134.16238706
 56555.71884035 56766.94559768]


((20000,), (2000,))

In [10]:
idx_atac, idx_expr

(array([ 1913, 37731, 56128, ..., 46390, 26010, 71830]),
 array([13881, 16527, 11934, ..., 13744, 13992,  4232]))

# Gene-peak interaction detection

In [11]:
'''

file_path = '../data/'
dataset = 'E13_5-S1'
atac = sc.read_h5ad(file_path + dataset + '_atac' + '.h5ad')
atac.X = scipy.sparse.csc_matrix(atac.X)
atac.obs['label'] = atac.obs['Annotation_for_Combined']
 
expr = sc.read_h5ad(file_path + dataset + '_expr' + '.h5ad')
expr.X = scipy.sparse.csc_matrix(expr.X)
expr.obs['label'] = expr.obs['Annotation_for_Combined']
expr.var_names_make_unique()

barcode_set1 = set(atac.obs_names)
barcode_set2 = set(expr.obs_names)

common_barcodes = barcode_set1.intersection(barcode_set2)

atac = atac[atac.obs_names.isin(common_barcodes), :]
expr = expr[expr.obs_names.isin(common_barcodes), :]
expr = expr[atac.obs_names]

'''

atac_20000 = atac[:,idx_atac]
expr_2000 = expr[:,idx_expr]
atac_20000,expr_2000

(View of AnnData object with n_obs × n_vars = 5274 × 20000
     obs: 'sample_id', 'slice_id', 'class_label', 'subclass', 'label', 'cell_id', 'centroid_x', 'centroid_y', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_5_genes', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_50_genes', 'uniform_density', 'rna_count_based_density'
     var: 'chrom', 'start', 'end', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'is_training'
     uns: 'files', 'gearyC', 'spatial_neighbors'
     obsm: 'spatial'
     obsp: 'spatial_connectivities', 'spatial_distances',
 View of AnnData object with n_obs × n_vars = 5274 × 2000
     obs: 'sample_id', 'slice_id', 'class_label', 'subclass', 'label', 'cell_id', 'centroid_x', 'centroid_y', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_5_genes', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_50_genes', 'un

In [12]:
sc.pp.normalize_total(expr_2000, target_sum=10000)
sc.pp.log1p(expr_2000)
sc.pp.scale(expr_2000)
expr_2000

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/scanpy/preprocessing/_normalization.py:169: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


AnnData object with n_obs × n_vars = 5274 × 2000
    obs: 'sample_id', 'slice_id', 'class_label', 'subclass', 'label', 'cell_id', 'centroid_x', 'centroid_y', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_5_genes', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_50_genes', 'uniform_density', 'rna_count_based_density', 'squidpy_domains', 'leiden'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'is_training', 'chrom', 'start', 'end', 'mean', 'std'
    uns: 'gearyC', 'leiden', 'leiden_colors', 'neighbors', 'overlap_genes', 'pca', 'spatial_neighbors', 'squidpy_domains_colors', 'subclass_colors', 'training_genes', 'umap', 'log1p'
    obsm: 'X_pca', 'X_umap', 'spatial', 'tangram_ct_pred'
    varm: 'PCs'
    obsp: 'connectivities', 'distances', 'spatial_connectivities', 'spatial_dis

In [13]:
count_mat = atac_20000.X.toarray().T

In [14]:
tf_mat = 1.0 * count_mat / np.tile(np.sum(count_mat,axis=0), (count_mat.shape[0],1))

In [15]:
ATAC_count = np.log(1 + np.multiply(1e4*tf_mat,  np.tile((1.0 * count_mat.shape[1] / np.sum(count_mat,axis=1)).reshape(-1,1), (1,count_mat.shape[1]))))

In [16]:
atac_20000.X = scipy.sparse.csc_matrix(ATAC_count.T)

In [17]:
adata = anndata.concat([atac_20000, expr_2000], axis=1)
adata.obs['label'] = list(atac_20000.obs['label'])
adata.obsm['spatial'] = atac_20000.obsm['spatial']
adata

AnnData object with n_obs × n_vars = 5274 × 22000
    obs: 'label'
    var: 'chrom', 'start', 'end', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'is_training'
    obsm: 'spatial'

In [18]:
adata.X = scipy.sparse.csc_matrix(adata.X)

ATAC_count_test = scale(adata.X.toarray())
count = ATAC_count_test.copy()
print(count.shape)
    
similarity_matrix_acb = np.zeros([count.shape[0], count.shape[0]])
similarity_matrix_spatial = np.zeros([count.shape[0], count.shape[0]])

distance_matrix = np.zeros([count.shape[0], count.shape[0]])
print(distance_matrix.shape)

count = PCA(n_components=pc,random_state=int(seed_base*1000)).fit_transform(count)
print(count.shape)
if similarity == 'Jaccard':
    if distance == 'euclidean':
        diff = count[:, np.newaxis, :] - count[np.newaxis, :, :]
        distance_matrix = np.linalg.norm(diff, axis=2)
        np.fill_diagonal(distance_matrix, np.inf)
    elif distance == 'cosine':
        count_norm = np.linalg.norm(count, axis=1, keepdims=True)
        dot_product_matrix = np.dot(count, count.T)
        distance_matrix = 1 - dot_product_matrix / (count_norm * count_norm.T)
        np.fill_diagonal(distance_matrix, np.inf)

distance_matrix_acb = np.copy(distance_matrix)

spatial_data = adata.obsm['spatial']
diff = spatial_data[:, np.newaxis, :] - spatial_data[np.newaxis, :, :]
distance_matrix = np.linalg.norm(diff, axis=2)
np.fill_diagonal(distance_matrix, np.inf)
distance_matrix_spatial = np.copy(distance_matrix)

similarity_matrix_acb = np.zeros([count.shape[0], count.shape[0]])
similarity_matrix_spatial = np.zeros([count.shape[0], count.shape[0]])
if spmethod == 'threshold':
    min_dist = np.min(distance_matrix_spatial)
    if sp_dist == 'const':
        similarity_matrix_spatial = np.array(distance_matrix_spatial <= neighbor * min_dist,dtype=int)
    elif sp_dist == 'recip':
        similarity_matrix_spatial = min_dist / distance_matrix_spatial
        similarity_matrix_spatial = similarity_matrix_spatial * (distance_matrix_spatial <= neighbor * min_dist)
    else:
        similarity_matrix_spatial = (min_dist / distance_matrix_spatial)**2
        similarity_matrix_spatial = similarity_matrix_spatial * (distance_matrix_spatial <= neighbor * min_dist)

elif spmethod == 'SNN':
    neighbor_index = np.argsort(distance_matrix_spatial, axis=1)[:,0:k]
    if similarity == "Jaccard":
        for i in range(count.shape[0]):
            for j in range(i):
                intersect_num = len(np.intersect1d(neighbor_index[i,:], neighbor_index[j,:]))
                similarity_matrix_spatial[i][j] = 1.0*intersect_num/(2*k-intersect_num)
                similarity_matrix_spatial[j][i] = 1.0*intersect_num/(2*k-intersect_num)
if similarity == "Jaccard":
    neighbor_index = np.argsort(distance_matrix_acb, axis=1)[:,0:k]
    for i in range(count.shape[0]):
        for j in range(i):
            intersect_num = len(np.intersect1d(neighbor_index[i,:], neighbor_index[j,:]))
            similarity_matrix_acb[i][j] = 1.0*intersect_num/(2*k-intersect_num)
            similarity_matrix_acb[j][i] = 1.0*intersect_num/(2*k-intersect_num)
elif similarity == "cosine":
    similarity_matrix_acb = sklearn.metrics.pairwise.cosine_similarity(count)
    for i in range(count.shape[0]):
        similarity_matrix_acb[i][i] = -float('inf')
    neighbor_index = np.argsort(similarity_matrix_acb, axis=1)[:,0:(count.shape[0]-k)]
    for i in range(count.shape[0]):
        similarity_matrix_acb[i,neighbor_index[i,:]] = 0
    print(similarity_matrix_acb[0,0])

similarity_matrix = (1 - r) * similarity_matrix_acb + r*similarity_matrix_spatial


print('compute scores')
scores = np.zeros(adata.n_vars)
X_processed = ATAC_count_test

temp_matrix = np.matmul(similarity_matrix, X_processed)
scores = np.sum(X_processed * temp_matrix, axis=0)
sorted_index = np.argsort(scores)
idx = sorted_index[::-1]
print('scores:')
print(scores[idx])

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:258: UserWarning: Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.
  warnings.warn(
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:277: UserWarning: Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 
  warnings.warn(


(5274, 22000)
(5274, 5274)
(5274, 10)
0.0
compute scores
scores:
[57233.2451587  55897.8060571  55289.80487738 ...  5182.83585593
  4903.13022803  4483.88580833]


In [19]:
start_time = time.time()
pdist_ = peak_modules_(selected_peaks_data=X_processed, similarity_matrix=similarity_matrix, method='complete')
pdist_ = (pdist_+pdist_.T)/2
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Running time: {elapsed_time:.6f} seconds")
print("Peak modules calculation has been done.")

pdist_.shape

Running time: 192.393840 seconds
Peak modules calculation has been done.


(22000, 22000)

In [20]:
np.savez("cached_data.npz", idx_atac=idx_atac, idx_expr=idx_expr, pdist=pdist_)

In [21]:
data = np.load("cached_data.npz")
idx_atac = data["idx_atac"]
idx_expr = data["idx_expr"]
pdist_ = data["pdist"]

In [22]:
g_p_similarity = pdist_[20000:,:20000]

## The most likely gene and peak

In [23]:
index = np.unravel_index(np.argmax(g_p_similarity, axis=None), g_p_similarity.shape)
index

(1935, 19998)

In [ ]:
accs = expr_2000.X[:,index[0]]
coord_x = np.array(atac_20000.obsm['spatial'][:,0])
coord_y = np.array(atac_20000.obsm['spatial'][:,1])
accs = np.array(accs)
accs = np.squeeze(accs)
plt.scatter(coord_x, coord_y, c=accs, s=3, cmap='Blues',vmin=-3, vmax=3)
plt.colorbar()
plt.show()

accs = scale(atac_20000.X.toarray())[:,index[1]]
accs = np.array(accs)
accs = np.squeeze(accs)
plt.scatter(coord_x, coord_y, c=accs, s=3, cmap='Blues',vmin=-3, vmax=3)
plt.colorbar()
plt.show()

## Gene to peak

In [ ]:
expr_matrix = expr_2000.X
atac_matrix = scale(atac_20000.X.toarray())
expr_matrix.shape, atac_matrix.shape

In [ ]:
a = anndata.AnnData(scipy.sparse.csc_matrix(atac_matrix))
a.obs_names = list(atac.obs_names)
a.obs['label'] = list(atac.obs['label'])
a.obsm['spatial'] = atac.obsm['spatial']

b = anndata.AnnData(scipy.sparse.csc_matrix(expr_matrix))
b.obs_names = list(expr.obs_names)
b.obs['label'] = list(expr.obs['label'])
b.obsm['spatial'] = expr.obsm['spatial']

result = np.zeros_like(g_p_similarity)
for col in range(g_p_similarity.shape[1]):
    indices = np.argpartition(g_p_similarity[:, col], -20)[-20:]
    result[indices, col] = g_p_similarity[indices, col]
    
    indices = np.argpartition(g_p_similarity[:, col],5)[:5]
    result[indices, col] = g_p_similarity[indices, col]
result.shape

In [ ]:
atac_matrix_pred = expr_matrix @ result
atac_matrix_pred = scale(atac_matrix_pred)
atac_matrix_pred.shape

In [ ]:
accs = atac_matrix[:,19998]
accs = np.array(accs)
accs = np.squeeze(accs)
plt.scatter(coord_x, coord_y, c=accs, s=3, cmap='Blues',vmin=-3, vmax=3)
plt.colorbar()
plt.show()

accs = atac_matrix_pred[:,19998]
accs = np.array(accs)
accs = np.squeeze(accs)
plt.scatter(coord_x, coord_y, c=accs, s=3, cmap='Blues',vmin=-3, vmax=3)
plt.colorbar()
plt.show()

## Peak to gene

In [ ]:
g_p_similarity_T = g_p_similarity.T
result = np.zeros_like(g_p_similarity_T)
for col in range(g_p_similarity_T.shape[1]):
    indices = np.argpartition(g_p_similarity_T[:, col], -20)[-20:]
    result[indices, col] = g_p_similarity_T[indices, col]
    
    indices = np.argpartition(g_p_similarity_T[:, col],5)[:5]
    result[indices, col] = g_p_similarity_T[indices, col]
result.shape

In [ ]:
expr_matrix_pred = atac_matrix @ result
expr_matrix_pred = scale(expr_matrix_pred)
expr_matrix_pred.shape

In [ ]:
accs = expr_matrix[:,1999]
accs = np.array(accs)
accs = np.squeeze(accs)
plt.scatter(coord_x, coord_y, c=accs, s=3, cmap='Blues',vmin=-3, vmax=3)
plt.colorbar()
plt.show()

accs = expr_matrix_pred[:,1999]
accs = np.array(accs)
accs = np.squeeze(accs)
plt.scatter(coord_x, coord_y, c=accs, s=3, cmap='Blues',vmin=-3, vmax=3)
plt.colorbar()
plt.show()